# FITMAX — Causa Raiz de Atraso (ICP, coorte 2026-06-01)

Executa as 3 queries do projeto, salva CSVs e gera o relatório `root_cause_report.md`.

In [1]:
from google.cloud import bigquery
import pandas as pd
from pathlib import Path

PROJECT_ID = 'insider-data-lake'
client = bigquery.Client(project=PROJECT_ID)
BASE_DIR = Path('/Users/insider/LA_Coding_Projects/4_Analysis/fitmax_mp_atrasada')
ANALYSIS_DATE = '2026-06-01'

print('Connected:', PROJECT_ID)

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Connected: insider-data-lake


## 1. Detalhe das OPs apontadas (OPF45N513 e OPF54N81)

In [2]:
sql_target = (BASE_DIR / 'target_ops_details.sql').read_text()
df_target = client.query(sql_target).to_dataframe()
df_target.to_csv(BASE_DIR / 'target_ops_details.csv', index=False)
print(f'{len(df_target)} linhas | salvo em target_ops_details.csv')
df_target

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


2 linhas | salvo em target_ops_details.csv


,order_code,status,production_stage,cycle_name,sc_supplier_name,current_production_stages,dt_min_entry_warehouse,dt_planned_entry_warehouse,dt_reviewed_entry_warehouse,dt_planned_production_start,...,is_mp_to_expected_end_lt_45_days,is_productive_lt_45_days,quality_supplier_name,audit_count,dt_first_audit_completed,first_audit_result_standardized,first_audit_deliberation_standardized,first_audit_defective_rate,pieces_rejected_in_first_audit_insider,quality_query_execution_date
0,OPF45N513,cut_fabric_and_sewing_process,cut_fabric_and_sewing_process,24-16-s3,FITMAX,cut_fabric_and_sewing_process,NaT,2026-05-15,2026-06-19,2025-04-21,...,<NA>,False,None,<NA>,NaT,None,None,NaN,<NA>,NaT
1,OPF54N81,cut_fabric_and_sewing_process,cut_fabric_and_sewing_process,24-09-s3,FITMAX,cut_fabric_and_sewing_process,NaT,2026-05-25,2026-06-30,2024-10-07,...,False,False,None,<NA>,NaT,None,None,NaN,<NA>,NaT


## 2. Todas as OPs Fitmax detratoras (janela 90 dias)

In [3]:
sql_details = (BASE_DIR / 'details.sql').read_text()
df_details = client.query(sql_details).to_dataframe()
df_details.to_csv(BASE_DIR / 'details.csv', index=False)
print(f'{len(df_details)} linhas | salvo em details.csv')
df_details

3 linhas | salvo em details.csv


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,order_code,cycle_name,sc_supplier_name,current_production_stages,dt_min_entry_warehouse,dt_planned_entry_warehouse,dt_reviewed_entry_warehouse,dt_planned_production_start,planned_quantity_op,received_quantity_op,...,supplier_legal_name,planned_production_delivery_date,expected_production_delivery_date,expected_fabric_receiving_date,real_fabric_receiving_date,fabric_receiving_delay_days,mp_to_expected_end_lead_time_days,productive_lead_time_days,fabric_receiving_status,days_overdue
0,OPF54N204,C092025,FITMAX,cut_fabric_and_sewing_process,NaT,2026-04-01,2026-06-19,2025-07-28,1035,<NA>,...,FITMAX LTDA,2026-04-01,2026-06-19,2025-08-28,NaT,<NA>,<NA>,66,SEM_DATA_REAL,61
1,OPF45N513,24-16-s3,FITMAX,cut_fabric_and_sewing_process,NaT,2026-05-15,2026-06-19,2025-04-21,702,<NA>,...,FITMAX LTDA,2026-05-15,2026-06-19,2025-08-07,NaT,<NA>,<NA>,102,SEM_DATA_REAL,17
2,OPF54N81,24-09-s3,FITMAX,cut_fabric_and_sewing_process,NaT,2026-05-25,2026-06-30,2024-10-07,928,<NA>,...,FITMAX LTDA,2026-05-25,2026-06-30,NaT,2024-10-07,<NA>,631,77,SEM_DATA_ESPERADA,7


## 3. Tabela de causa raiz por volume

In [4]:
sql_summary = (BASE_DIR / 'root_cause_summary.sql').read_text()
df_summary = client.query(sql_summary).to_dataframe()
df_summary.to_csv(BASE_DIR / 'root_cause_summary.csv', index=False)
print(f'{len(df_summary)} linhas | salvo em root_cause_summary.csv')
df_summary

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


0 linhas | salvo em root_cause_summary.csv


,causa_raiz,qtd_ops,volume_pecas,pct_do_volume_total


## 4. Geração do relatório Markdown

In [5]:
import math

# ── helpers ────────────────────────────────────────────────────────────────
def fmt_date(val):
    if pd.isna(val) or val is None:
        return '—'
    return str(val)[:10]

def fmt_int(val):
    if pd.isna(val) or val is None:
        return '—'
    return f'{int(val):,}'.replace(',', '.')

def fmt_pct(val):
    if pd.isna(val) or val is None:
        return '—'
    return f'{float(val)*100:.2f}%'

def bool_flag(val):
    if pd.isna(val) or val is None:
        return '—'
    return '✅ Sim' if val else '❌ Não'

def quality_label(result, deliberation):
    if pd.isna(result) or result is None:
        return 'Sem registro'
    if result == 'qualita_approved':
        return 'Aprovada'
    if result == 'qualita_rejected':
        if deliberation == 'insider_approved':
            return 'Reprovada — entrega autorizada'
        if deliberation == 'insider_rejected':
            return 'Reprovada — corrigir/reauditar'
        return f'Reprovada ({deliberation})'
    return result

def mp_status_label(status, real, expected, delay):
    if status == 'SEM_DATA_REAL':
        return f'**MP sem data real de recebimento** (esperada: `{fmt_date(expected)}`)'
    if status == 'SEM_DATA_ESPERADA':
        return f'**Sem data esperada** (real: `{fmt_date(real)}`)'
    if status == 'ATRASADA':
        return f'`{fmt_date(real)}` (esperada: `{fmt_date(expected)}`) — **{int(delay)} dias de atraso**'
    if status == 'NO_PRAZO':
        return f'`{fmt_date(real)}` — no prazo'
    if status == 'ADIANTADA':
        return f'`{fmt_date(real)}` (esperada: `{fmt_date(expected)}`) — adiantada'
    return '—'

# ── contexto geral ──────────────────────────────────────────────────────────
total_ops = len(df_details)
total_pecas = int(df_details['planned_quantity_op'].sum()) if total_ops > 0 else 0
ops_mp_atrasada   = int(df_details['fabric_receiving_status'].eq('ATRASADA').sum())
ops_sem_data_real = int(df_details['fabric_receiving_status'].eq('SEM_DATA_REAL').sum())
ops_sem_data_esp  = int(df_details['fabric_receiving_status'].eq('SEM_DATA_ESPERADA').sum())
total_pecas_informadas = 702 + 928  # OPs apontadas na análise

# ── universo real (3 OPs) ─────────────────────────────────────────────────────
# ordena pelo atraso
df_det_sorted = df_details.sort_values('days_overdue', ascending=False).reset_index(drop=True)

def all_ops_table():
    lines = []
    for _, r in df_det_sorted.iterrows():
        lines.append(
            f"| `{r['order_code']}` "
            f"| {r['cycle_name']} "
            f"| {fmt_int(r['planned_quantity_op'])} "
            f"| `{fmt_date(r['dt_planned_entry_warehouse'])}` "
            f"| `{fmt_date(r['dt_reviewed_entry_warehouse'])}` "
            f"| {fmt_int(r['days_overdue'])} "
            f"| `{r['fabric_receiving_status']}` "
            f"| {fmt_int(r['productive_lead_time_days'])} |"
        )
    return '\n'.join(lines)

# ── detalhes por OP apontada ─────────────────────────────────────────────────
OP_META = {
    'OPF45N513': {'produto': 'Shorts Esportivo Serotonin Feminino', 'prazo_orig': '2026-05-15'},
    'OPF54N81':  {'produto': 'Performance T-shirt 2.0 Masculino',   'prazo_orig': '2026-05-25'},
}

def get_op_target(op_code):
    rows = df_target[df_target['order_code'] == op_code]
    if rows.empty:
        return None
    return rows.iloc[0]

def op_block(op_code):
    meta   = OP_META.get(op_code, {})
    row    = get_op_target(op_code)
    produto = meta.get('produto', '—')

    if row is None:
        return f"### {op_code} — {produto}\n\n> ⚠️ OP não encontrada na query. Verificar código no BigQuery.\n"

    r = row
    det_row = df_details[df_details['order_code'] == op_code]
    mp_st = det_row['fabric_receiving_status'].values[0] if not det_row.empty else '—'
    delay = r.get('fabric_receiving_delay_days')
    lt_mp  = r.get('mp_to_expected_production_end_lead_time_days')
    lt_pr  = r.get('productive_lead_time_days')
    days_ov = det_row['days_overdue'].values[0] if not det_row.empty else '—'

    return f"""### `{op_code}` — {produto}

| Campo | Valor |
|---|---|
| **Peças planejadas** | {fmt_int(r.get('planned_quantity_op'))} |
| **Prazo original (SC)** | `{fmt_date(r.get('dt_planned_entry_warehouse'))}` |
| **Prazo revisado** | `{fmt_date(r.get('dt_reviewed_entry_warehouse'))}` |
| **Atraso s/ prazo original** | {days_ov} dias |
| **Ciclo** | `{r.get('cycle_name', '—')}` |
| **Status Muninn** | `{r.get('status', '—')}` |
| **Estágio SC** | `{r.get('current_production_stages', '—')}` |
| **Início produção planejado** | `{fmt_date(r.get('dt_planned_production_start'))}` |

#### Matéria-Prima

| Campo | Valor |
|---|---|
| Status de recebimento de MP | `{mp_st}` |
| MP esperada | `{fmt_date(r.get('expected_fabric_receiving_date'))}` |
| MP real | `{fmt_date(r.get('real_fabric_receiving_date'))}` |
| Atraso de MP | {fmt_int(delay) + ' dias' if not pd.isna(delay) and delay is not None else '—'} |
| LT MP → entrega esperada | {fmt_int(lt_mp) + ' dias' if not pd.isna(lt_mp) and lt_mp is not None else '—'} |
| LT MP → fim < 45 dias? | {bool_flag(r.get('is_mp_to_expected_end_lt_45_days'))} |

#### Lead Time Produtivo

| Campo | Valor |
|---|---|
| LT produtivo (início → entrega esperada) | {fmt_int(lt_pr) + ' dias' if not pd.isna(lt_pr) and lt_pr is not None else '—'} |
| LT produtivo < 45 dias? | {bool_flag(r.get('is_productive_lt_45_days'))} |

#### Qualidade

| Campo | Valor |
|---|---|
| Resultado 1ª auditoria | {quality_label(r.get('first_audit_result_standardized'), r.get('first_audit_deliberation_standardized'))} |
| Data 1ª auditoria | `{fmt_date(r.get('dt_first_audit_completed'))}` |
| Taxa de defeitos | {fmt_pct(r.get('first_audit_defective_rate'))} |
| Peças rejeitadas (Insider) | {fmt_int(r.get('pieces_rejected_in_first_audit_insider'))} |
"""

# ── tabela de causa raiz ─────────────────────────────────────────────────────
def causa_raiz_table_md(df):
    if df.empty:
        return '| — | — | — | — |\n> ℹ️ Nenhuma causa raiz ativou no modelo padrão (ver análise abaixo).\n'
    lines = []
    for _, row in df.iterrows():
        lines.append(
            f"| {row['causa_raiz']} "
            f"| {fmt_int(row['qtd_ops'])} "
            f"| {fmt_int(row['volume_pecas'])} "
            f"| {fmt_pct(row['pct_do_volume_total'])} |"
        )
    return '\n'.join(lines)

# ── monta relatório ──────────────────────────────────────────────────────────
report = f"""# FITMAX — Relatório de Causa Raiz de Atraso

**Data de análise:** `{ANALYSIS_DATE}`
**Fornecedor:** FITMAX LTDA
**Contexto:** A Fitmax entrou no coorte de ICP na semana de {ANALYSIS_DATE} com indicador de **0%**.
As {fmt_int(total_pecas_informadas)} peças das OPs apontadas estão planejadas e nenhuma foi recebida no CD.

---

## Universo de OPs detratoras (janela 90 dias — de 2026-03-03 a {ANALYSIS_DATE})

| Métrica | Valor |
|---|---|
| Total de OPs detratoras Fitmax | {fmt_int(total_ops)} |
| Volume total (peças) | {fmt_int(total_pecas)} |
| OPs com MP atrasada (real > esperada) | {fmt_int(ops_mp_atrasada)} |
| OPs sem data real de MP | {fmt_int(ops_sem_data_real)} |
| OPs sem data esperada de MP | {fmt_int(ops_sem_data_esp)} |

> ⚠️ **Além das 2 OPs apontadas, há uma terceira OP detratora (`OPF54N204`) com 61 dias de atraso
> e 1.035 peças, não mencionada na análise original.**

### Detalhamento

| OP | Ciclo | Peças | Prazo Orig. | Prazo Revisado | Dias Atraso | Status MP | LT Produtivo |
|---|---|---:|---|---|---:|---|---:|
{all_ops_table()}

---

## Detalhe por OP apontada

{op_block('OPF45N513')}
---

{op_block('OPF54N81')}
---

## Tabela de Causa Raiz por Volume (modelo padrão)

| Causa Raiz | Qtd OPs | Volume (Peças) | % do Volume Total |
|---|---:|---:|---:|
{causa_raiz_table_md(df_summary)}

---

## Análise de Causa Raiz — Achados

O modelo padrão (atraso MP / LT curto / reprovação de qualidade) **não ativou nenhuma causa**
para as OPs Fitmax. Isso ocorre porque os campos de data de MP estão incompletos.
Os achados reais são:

### 1. MP sem registro de recebimento real (OPF45N513 e OPF54N204)

As OPs `OPF45N513` e `OPF54N204` têm `expected_fabric_receiving_date` preenchida
(respectivamente `2025-08-07` e `2025-08-28`) mas **`real_fabric_receiving_date` é nulo**
— ou seja, a MP **nunca foi registrada como recebida** no sistema.

Hipóteses:
- A MP realmente não chegou ao fornecedor (bloqueio de cadeia de suprimentos).
- A MP chegou mas o recebimento não foi lançado na ferramenta.

**Ação recomendada:** confirmar com a Fitmax e/ou time de MP se o tecido foi recebido
e, em caso positivo, solicitar lançamento retroativo da data real.

### 2. OPF54N81 — OP de ciclo longo sem data esperada de MP

A `OPF54N81` (ciclo `24-09-s3`, setembro/2024) tem `real_fabric_receiving_date = 2024-10-07`
— **MP recebida há mais de 7 meses** — mas a OP ainda está em `cut_fabric_and_sewing_process`.
O `expected_fabric_receiving_date` está nulo.

Com LT produtivo de **77 dias** (início → entrega esperada de 30/06/2026), a OP está dentro
do prazo *revisado*, mas com **7 dias de atraso sobre o prazo original** (2026-05-25).

A causa de atraso aqui não é MP nem qualidade: é **lead time de produção** — a OP
está em corte/costura há meses com o prazo original já ultrapassado.

### 3. Prazos revisados indicam reprogramação ativa

Todos os prazos revisados (`dt_reviewed_entry_warehouse`) estão no futuro:
- `OPF45N513`: revisado para `2026-06-19`
- `OPF54N81`: revisado para `2026-06-30`
- `OPF54N204`: revisado para `2026-06-19`

Isso confirma que as OPs foram reprogramadas. O ICP mede sobre o prazo original,
daí o indicador de 0% mesmo com as OPs ainda "dentro do prazo revisado" do fornecedor.

### 4. Qualidade — sem registro

Nenhuma das 3 OPs tem registro na `quality_inspection_data`. Pode indicar que as
peças ainda não chegaram à etapa de auditoria (estão em corte/costura) ou que a
auditoria ainda não foi registrada.

---

## Conclusão

| Causa Raiz | OPs afetadas | Volume (peças) | Observação |
|---|---|---:|---|
| MP sem data real (não registrada/não chegou) | OPF45N513, OPF54N204 | 1.737 | Principal hipótese; requer confirmação |
| Lead time de produção longo (ciclo set/24) | OPF54N81 | 928 | MP OK, atraso no processo produtivo |
| Reprovação de qualidade | Nenhuma | 0 | Sem dado disponível ainda |

**Causa dominante (por volume): MP sem data real de recebimento — 1.737 peças (65,1% do volume).**

---

## Observações de qualidade de dados

- `real_fabric_receiving_date` nulo ≠ confirmação de que a MP não chegou. Verificar
  com time operacional antes de classificar como "atraso de MP".
- `OPF54N81` tem `real_fabric_receiving_date = 2024-10-07` sem `expected_fabric_receiving_date`.
  Possível inconsistência de cadastro ou OP com histórico atípico.
- O campo `current_production_stages` reflete o estágio mais recente com defasagem de até 24h.

---

## Arquivos

- `target_ops_details.sql` / `.csv`: detalhe das OPs OPF45N513 e OPF54N81.
- `details.sql` / `.csv`: todas as OPs Fitmax detratoras na janela de 90 dias.
- `root_cause_summary.sql` / `.csv`: tabela de causa raiz por volume.
- `fitmax_relatorio.ipynb`: notebook de execução e geração deste relatório.
"""

output_path = BASE_DIR / 'root_cause_report.md'
output_path.write_text(report, encoding='utf-8')
print(f'Relatório salvo em: {output_path}')
print()
print(report)


Relatório salvo em: /Users/insider/LA_Coding_Projects/4_Analysis/fitmax_mp_atrasada/root_cause_report.md

# FITMAX — Relatório de Causa Raiz de Atraso

**Data de análise:** `2026-06-01`
**Fornecedor:** FITMAX LTDA
**Contexto:** A Fitmax entrou no coorte de ICP na semana de 2026-06-01 com indicador de **0%**.
As 1.630 peças das OPs apontadas estão planejadas e nenhuma foi recebida no CD.

---

## Universo de OPs detratoras (janela 90 dias — de 2026-03-03 a 2026-06-01)

| Métrica | Valor |
|---|---|
| Total de OPs detratoras Fitmax | 3 |
| Volume total (peças) | 2.665 |
| OPs com MP atrasada (real > esperada) | 0 |
| OPs sem data real de MP | 2 |
| OPs sem data esperada de MP | 1 |

> ⚠️ **Além das 2 OPs apontadas, há uma terceira OP detratora (`OPF54N204`) com 61 dias de atraso
> e 1.035 peças, não mencionada na análise original.**

### Detalhamento

| OP | Ciclo | Peças | Prazo Orig. | Prazo Revisado | Dias Atraso | Status MP | LT Produtivo |
|---|---|---:|---|---|---:|---|---:|
| `OPF